# 01 - Exploracion de datos: ModelNet10

Este notebook esta pensado para ejecutarse en **Google Colab** (GPU T4 gratuita).

Objetivo de este notebook (Hora 1 del plan): clonar el repo, descargar ModelNet10,
preprocesar un subconjunto de nubes de puntos y **verificar visualmente** que el
pipeline de `src/data.py` hace lo que creemos que hace, antes de construir el
modelo encima. Comprobar esto ahora evita depurar a ciegas mas tarde si algo
en el preprocesado estuviera mal (p.ej. una nube de puntos no centrada, o
clases mal mapeadas).

## 0. Setup: clonar el repo e instalar dependencias

Colab arranca con un runtime limpio en cada sesion, asi que hay que clonar el
repo y las dependencias que no vienen preinstaladas (`trimesh`) cada vez.
torch, numpy, matplotlib, scikit-learn ya vienen instalados en Colab por defecto.

In [ ]:
# Si se ejecuta en Colab, clona el repo y entra en la carpeta. Si ya se esta
# ejecutando localmente dentro del repo clonado, esta celda no hace falta.
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.isdir("3d_classifier"):
        !git clone https://github.com/marsaliborra/3d_classifier.git
    %cd 3d_classifier
    !pip install -q trimesh
    print("Ejecutando en Colab.")
else:
    print("Ejecutando localmente (asumo que ya estoy en la raiz del repo o en notebooks/).")

In [ ]:
# Este notebook vive en notebooks/, pero src/ esta un nivel por encima.
# Anadimos la raiz del repo al path para poder hacer `from src import data`.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data import (
    CLASSES,
    download_modelnet10,
    list_off_files,
    get_dataset_arrays,
)

DATA_ROOT = REPO_ROOT / "data"
N_POINTS = 1024  # mismo numero de puntos que usa el paper original de PointNet

## 1. Descargar ModelNet10

`download_modelnet10` es idempotente: si la carpeta `data/ModelNet10` ya existe
(por ejemplo porque ya se ejecuto esta celda antes en la misma sesion de Colab),
no vuelve a descargar los ~450MB.

In [ ]:
dataset_dir = download_modelnet10(root=str(DATA_ROOT))
dataset_dir

## 2. Conteo de muestras por clase

Antes de entrenar nada, conviene saber si el dataset esta balanceado entre
clases. Si una clase tuviera muchas menos muestras que las demas, la accuracy
global podria ser enganosa (el modelo podria ignorarla casi por completo y aun
asi obtener buena accuracy media) y habria que tenerlo en cuenta al leer la
matriz de confusion mas adelante.

In [ ]:
# Version mas eficiente del conteo anterior (list_off_files recorre el disco
# cada vez que se llama; para 10 clases x 2 splits no importa, pero lo hacemos
# una sola vez por split para tener los datos listos para graficar).
train_samples = list_off_files(dataset_dir, "train")
test_samples = list_off_files(dataset_dir, "test")

train_counts = [sum(1 for _, c in train_samples if c == cls) for cls in CLASSES]
test_counts = [sum(1 for _, c in test_samples if c == cls) for cls in CLASSES]

x = np.arange(len(CLASSES))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width / 2, train_counts, width, label="train")
ax.bar(x + width / 2, test_counts, width, label="test")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_ylabel("numero de muestras")
ax.set_title("ModelNet10: muestras por clase y split")
ax.legend()
plt.tight_layout()
plt.show()

train_samples = list_off_files(dataset_dir, "train")
test_samples = list_off_files(dataset_dir, "test")

train_counts = [sum(1 for _, c in train_samples if c == cls) for cls in CLASSES]
test_counts = [sum(1 for _, c in test_samples if c == cls) for cls in CLASSES]

for split_name, counts in [("train", train_counts), ("test", test_counts)]:
    print(f"--- {split_name} (total={sum(counts)}) ---")
    for cls, n in zip(CLASSES, counts):
        print(f"  {cls:12s}: {n}")

x = np.arange(len(CLASSES))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width / 2, train_counts, width, label="train")
ax.bar(x + width / 2, test_counts, width, label="test")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_ylabel("numero de muestras")
ax.set_title("ModelNet10: muestras por clase y split")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
train_points, train_labels = get_dataset_arrays(dataset_dir, "train", n_points=N_POINTS, cache_dir=str(DATA_ROOT))
test_points, test_labels = get_dataset_arrays(dataset_dir, "test", n_points=N_POINTS, cache_dir=str(DATA_ROOT))

print("train_points:", train_points.shape, train_points.dtype)
print("train_labels:", train_labels.shape, train_labels.dtype)
print("test_points: ", test_points.shape)
print("test_labels: ", test_labels.shape)

## 4. Sanity check de la normalizacion

Verificacion directa de que `normalize_point_cloud` hizo lo que dice que hace:
para cada nube, el centroide deberia estar en (0,0,0) (dentro de error de
float32) y la distancia maxima al origen deberia ser exactamente 1.0.

Esto no es una comprobacion cosmetica: si esto fallara, significaria que el
modelo esta viendo objetos a escalas y posiciones inconsistentes entre si, lo
cual romperia la premisa de que la red aprende FORMA y no posicion/escala.

In [ ]:
sample_idx = 0
cloud = train_points[sample_idx]  # (N_POINTS, 3)

centroid = cloud.mean(axis=0)
max_radius = np.linalg.norm(cloud, axis=1).max()

print(f"Clase: {CLASSES[train_labels[sample_idx]]}")
print(f"Centroide (deberia ser ~[0,0,0]): {centroid}")
print(f"Radio maximo (deberia ser ~1.0): {max_radius:.4f}")

## 5. Visualizar nubes de puntos de ejemplo

Inspeccion visual de 4 muestras aleatorias del set de entrenamiento. El
objetivo es confirmar "a ojo" que las formas son reconocibles (una silla se
parece a una silla) despues de todo el proceso de muestreo y normalizacion -
una comprobacion cualitativa que ningun assert numerico sustituye del todo.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (necesario para projection='3d')

rng = np.random.default_rng(42)
sample_indices = rng.choice(len(train_points), size=4, replace=False)

fig = plt.figure(figsize=(16, 4))
for i, idx in enumerate(sample_indices):
    ax = fig.add_subplot(1, 4, i + 1, projection="3d")
    cloud = train_points[idx]
    ax.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2], s=2, alpha=0.6)
    ax.set_title(CLASSES[train_labels[idx]])
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_zlim(-1, 1)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Siguiente paso

Con los datos verificados (balance de clases razonable, normalizacion
correcta, formas reconocibles visualmente), el siguiente notebook/script
(`src/model.py` + entrenamiento) construye la arquitectura PointNet
simplificada sobre estos mismos tensores.